## Setup

In [1]:
import pandas as pd

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
except ImportError:
    plt = None
    sns = None

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

from pgvectordb import IndexType, RAGEvaluator, create_sample_evaluation_dataset, pgVectorDB

if sns is not None and plt is not None:
    sns.set_style("whitegrid")
    plt.rcParams["figure.figsize"] = (12, 6)

## Database Configuration

In [2]:
# Database connection
DB_HOST = "localhost"
DB_PORT = "9002"
DB_NAME = "postgres"
DB_USER = "user"
DB_PASSWORD = "root"

CONNECTION_STRING = f"postgresql+asyncpg://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
print(f"Database: {DB_HOST}:{DB_PORT}/{DB_NAME}")

Database: localhost:9002/postgres


## 1. Create Evaluation Dataset

We'll create a comprehensive dataset with queries across multiple categories and well-defined ground truth.

In [3]:
# Create evaluation dataset
eval_dataset = create_sample_evaluation_dataset()

print("Evaluation Dataset:")
print(f"  Total queries: {len(eval_dataset)}")
print("\nSample queries:")
for i in range(min(5, len(eval_dataset))):
    query, relevant, meta = eval_dataset[i]
    print(f"  {i + 1}. {query[:60]}... ({len(relevant)} relevant docs)")

# Save dataset
eval_dataset.save("evaluation_dataset.json")
print("\n✅ Dataset saved to 'evaluation_dataset.json'")

Evaluation Dataset:
  Total queries: 21

Sample queries:
  1. How to implement a binary search tree in Python?... (3 relevant docs)
  2. What are Python decorators and how do they work?... (2 relevant docs)
  3. Explain async/await in Python... (4 relevant docs)
  4. What is transformer architecture in deep learning?... (3 relevant docs)
  5. How does gradient descent optimization work?... (4 relevant docs)

✅ Dataset saved to 'evaluation_dataset.json'


## 2. Create Document Corpus

Generate a realistic corpus of 100 documents that match the evaluation queries.

In [4]:
def create_evaluation_corpus() -> list[Document]:
    """
    Create a corpus of 100 documents for evaluation.
    Documents are designed to match the evaluation dataset queries.
    """
    documents = []

    # Programming documents (doc_0 to doc_24)
    programming_topics = [
        "Binary search trees are fundamental data structures in Python with O(log n) search complexity.",
        "Python decorators are functions that modify the behavior of other functions using @ syntax.",
        "Async/await in Python enables concurrent programming using asyncio coroutines.",
        "List comprehensions provide concise syntax for creating lists in Python.",
        "Python generators use yield to create memory-efficient iterators.",
        "Object-oriented programming in Python supports inheritance, polymorphism, and encapsulation.",
        "Context managers in Python handle resource allocation with __enter__ and __exit__ methods.",
        "Python metaclasses allow customization of class creation behavior.",
        "Type hints improve code clarity and enable static type checking in Python.",
        "Lambda functions provide anonymous function syntax for simple operations.",
        "Python's Global Interpreter Lock (GIL) affects multithreading performance.",
        "Virtual environments isolate Python dependencies per project.",
        "Python package management uses pip and requirements.txt files.",
        "Exception handling with try/except prevents program crashes.",
        "Python's data model defines how objects behave using magic methods.",
        "Decorators can be chained to apply multiple transformations to functions.",
        "Binary search trees support efficient insertion, deletion, and search operations.",
        "The asyncio library provides event loop for asynchronous programming.",
        "Python closures capture variables from enclosing scope.",
        "Design patterns like Factory and Singleton are commonly used in Python.",
        "Python's collections module provides specialized data structures.",
        "Understanding async/await requires knowledge of event loops and coroutines.",
        "Python memory management uses reference counting and garbage collection.",
        "Testing in Python uses unittest, pytest, and mock libraries.",
        "Binary trees can be balanced using AVL or Red-Black tree algorithms.",
    ]

    for i, content in enumerate(programming_topics):
        doc = Document(
            page_content=content,
            metadata={
                "doc_id": i,
                "category": "programming",
                "language": "Python",
                "difficulty": ["easy", "medium", "advanced"][i % 3],
                "year": 2020 + (i % 5),
            },
        )
        documents.append(doc)

    # AI/ML documents (doc_25 to doc_44)
    ai_topics = [
        "Transformer architecture revolutionized NLP with self-attention mechanisms.",
        "Gradient descent optimizes neural networks by minimizing loss functions.",
        "Convolutional neural networks excel at image recognition tasks.",
        "Recurrent neural networks process sequential data with hidden states.",
        "Attention mechanisms allow models to focus on relevant input parts.",
        "Transfer learning leverages pre-trained models for new tasks.",
        "Regularization techniques prevent overfitting in neural networks.",
        "Backpropagation computes gradients for neural network training.",
        "BERT uses bidirectional transformers for language understanding.",
        "GPT models generate text using autoregressive transformers.",
        "Training neural networks requires careful hyperparameter tuning.",
        "Batch normalization improves training stability and speed.",
        "Dropout randomly deactivates neurons to prevent overfitting.",
        "Learning rate scheduling adjusts optimization speed during training.",
        "Early stopping prevents overfitting by monitoring validation loss.",
        "Data augmentation increases training data diversity.",
        "Gradient descent variants include SGD, Adam, and RMSprop.",
        "Neural network best practices include proper initialization and normalization.",
        "Embeddings convert categorical data to dense vector representations.",
        "Fine-tuning adapts pre-trained models to specific domains.",
    ]

    for i, content in enumerate(ai_topics, start=25):
        doc = Document(
            page_content=content,
            metadata={
                "doc_id": i,
                "category": "ai",
                "language": "Python",
                "difficulty": ["medium", "advanced"][i % 2],
                "year": 2020 + (i % 5),
            },
        )
        documents.append(doc)

    # Database documents (doc_45 to doc_59)
    db_topics = [
        "PostgreSQL query optimization uses EXPLAIN ANALYZE for performance tuning.",
        "Database indexes improve query performance using B-tree structures.",
        "ACID properties ensure database transaction reliability.",
        "Normalization reduces data redundancy in relational databases.",
        "SQL joins combine data from multiple tables efficiently.",
        "Database sharding distributes data across multiple servers.",
        "Connection pooling reduces database connection overhead.",
        "PostgreSQL VACUUM maintains database performance.",
        "Indexes trade write performance for faster reads.",
        "Transactions ensure data consistency with COMMIT and ROLLBACK.",
        "Query planning optimizes execution with cost-based decisions.",
        "Foreign keys maintain referential integrity in databases.",
        "Database replication provides high availability and fault tolerance.",
        "Materialized views cache query results for faster access.",
        "ACID transactions guarantee atomicity, consistency, isolation, and durability.",
    ]

    for i, content in enumerate(db_topics, start=45):
        doc = Document(
            page_content=content,
            metadata={
                "doc_id": i,
                "category": "database",
                "language": "SQL",
                "difficulty": ["easy", "medium", "advanced"][i % 3],
                "year": 2020 + (i % 5),
            },
        )
        documents.append(doc)

    # Web, DevOps, Security, Cloud documents (doc_60 to doc_99)
    other_topics = [
        "FastAPI builds high-performance REST APIs with automatic OpenAPI documentation.",
        "JWT authentication uses signed tokens for stateless authentication.",
        "React hooks provide functional components with state and lifecycle features.",
        "Docker containers package applications with dependencies for consistent deployment.",
        "GitHub Actions automates CI/CD workflows with YAML configuration.",
        "Kubernetes orchestrates containerized applications at scale.",
        "SQL injection attacks exploit unsanitized database queries.",
        "OAuth 2.0 provides delegated authorization for third-party applications.",
        "AWS Lambda runs serverless functions without managing servers.",
        "Microservices architecture splits applications into independent services.",
        "GraphQL provides flexible API queries compared to REST.",
        "WebSockets enable real-time bidirectional communication.",
        "CORS policies control cross-origin resource access in browsers.",
        "Service mesh manages microservice communication and observability.",
        "Infrastructure as Code uses Terraform and CloudFormation.",
        "Blue-green deployment minimizes downtime during updates.",
        "Rate limiting prevents API abuse and ensures fair usage.",
        "Content Security Policy prevents XSS attacks in web applications.",
        "Load balancers distribute traffic across multiple servers.",
        "API gateways provide authentication, rate limiting, and routing.",
        "Container orchestration automates deployment and scaling.",
        "Serverless computing eliminates server management overhead.",
        "Zero-trust security validates every access request.",
        "Cloud-native applications use containerization and microservices.",
        "Event-driven architecture enables loose coupling between services.",
        "HTTPS encrypts data in transit using TLS certificates.",
        "Prometheus and Grafana monitor application performance.",
        "Helm manages Kubernetes application deployments.",
        "Redis provides in-memory caching for faster data access.",
        "Message queues enable asynchronous communication between services.",
        "Cloud storage services like S3 provide scalable object storage.",
        "CDNs distribute static content globally for faster access.",
        "API versioning maintains backward compatibility during updates.",
        "Circuit breakers prevent cascading failures in distributed systems.",
        "Observability combines logging, metrics, and tracing.",
        "Immutable infrastructure reduces configuration drift.",
        "Auto-scaling adjusts resources based on demand.",
        "Security scanning detects vulnerabilities in dependencies.",
        "API documentation improves developer experience with clear examples.",
        "Container registries store and distribute Docker images.",
    ]

    categories_cycle = ["web", "security", "cloud", "devops"]
    for i, content in enumerate(other_topics, start=60):
        doc = Document(
            page_content=content,
            metadata={
                "doc_id": i,
                "category": categories_cycle[(i - 60) % 4],
                "language": ["Python", "JavaScript", "Go", "TypeScript"][(i - 60) % 4],
                "difficulty": ["easy", "medium", "advanced"][i % 3],
                "year": 2020 + (i % 5),
            },
        )
        documents.append(doc)

    return documents


# Create corpus
corpus = create_evaluation_corpus()
print(f"Created corpus with {len(corpus)} documents")
print("\nSample documents:")
for i in range(3):
    print(f"  doc_{i}: {corpus[i].page_content[:60]}...")

Created corpus with 100 documents

Sample documents:
  doc_0: Binary search trees are fundamental data structures in Pytho...
  doc_1: Python decorators are functions that modify the behavior of ...
  doc_2: Async/await in Python enables concurrent programming using a...


## 3. Initialize RAG System

In [5]:
# Initialize embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2", model_kwargs={"device": "cpu"}
)

# Initialize RAG system with HNSW index
db = pgVectorDB(
    collection_name="evaluation_collection",
    embedding_model=embeddings,
    connection_string=CONNECTION_STRING,
    index_type=IndexType.HNSW,
)

# Initialize and add documents
await db.initialize(overwrite_existing=True)
doc_ids = await db.add_documents(corpus)
print(f"✅ Added {len(doc_ids)} documents")

# Create metadata indexes for filtered search
await db.create_metadata_index(["category", "difficulty", "year"])
print("✅ Created metadata indexes")

# Build vector index
await db.build_index()
print("✅ Built HNSW index")

# Get stats
stats = await db.get_stats()
print("\nDatabase stats:")
print(f"  Documents: {stats['document_count']}")
print(f"  Index: {stats.get('index_type', 'unknown')}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Added 100 documents
✅ Created metadata indexes
✅ Built HNSW index

Database stats:
  Documents: 100
  Index: hnsw


## 4. Evaluate All Search Methods

We'll test all 6 search methods and compare their retrieval metrics.

In [6]:
async def evaluate_search_method(
    method_name: str,
    search_function,
    queries: list[str],
    ground_truth: list[list[str]],
    k: int = 5,
) -> dict:
    """
    Evaluate a search method against ground truth.

    Args:
        method_name: Name of the search method
        search_function: Async function that performs search
        queries: List of query strings
        ground_truth: List of relevant doc IDs for each query
        k: Number of results to retrieve

    Returns:
        Dictionary with method name and results
    """
    print(f"\nEvaluating: {method_name}")

    retrieved_results = []

    for query in queries:
        results = await search_function(query, k=k)

        # Extract doc_id from query result metadata
        doc_ids = [
            f"doc_{r['metadata']['doc_id']}" for r in results if "doc_id" in r.get("metadata", {})
        ]
        retrieved_results.append(doc_ids)

    evaluator = RAGEvaluator(k=k)
    result = evaluator.evaluate(queries, retrieved_results, ground_truth)

    print(f"  ✅ Completed: Precision={result.precision:.3f}, Recall={result.recall:.3f}")

    return {"method": method_name, "results": result}


async def keyword_wrapper(query, k=5):
    return await db.query(query).keyword().fts().limit(k).to_list()


async def universal_keyword_wrapper(query, k=5):
    return await (
        db.query(query)
        .keyword()
        .fts()
        .universal(metadata_fields=["category", "difficulty", "tags"])
        .limit(k)
        .to_list()
    )


async def semantic_wrapper(query, k=5):
    return await db.query(query).semantic().limit(k).to_list()


def infer_category(query: str) -> str:
    query_lower = query.lower()
    if "database" in query_lower or "sql" in query_lower:
        return "database"
    if "transformer" in query_lower or "neural" in query_lower or "gradient" in query_lower:
        return "ai"
    if "api" in query_lower or "web" in query_lower or "react" in query_lower:
        return "web"
    if "docker" in query_lower or "kubernetes" in query_lower or "ci/cd" in query_lower:
        return "devops"
    if "security" in query_lower or "oauth" in query_lower or "injection" in query_lower:
        return "security"
    if "cloud" in query_lower or "aws" in query_lower or "lambda" in query_lower:
        return "cloud"
    return "programming"


async def metadata_keyword_wrapper(query, k=5):
    return await (
        db.query(query)
        .keyword()
        .fts()
        .where({"category": {"$eq": infer_category(query)}})
        .limit(k)
        .to_list()
    )


async def metadata_semantic_wrapper(query, k=5):
    return await (
        db.query(query)
        .semantic()
        .where({"category": {"$eq": infer_category(query)}})
        .limit(k)
        .to_list()
    )


async def hybrid_wrapper(query, k=5):
    return await (
        db.query(query).hybrid().fts().weights(semantic=0.6, keyword=0.4).limit(k).to_list()
    )


# Run evaluations for all methods
evaluation_results = []

for method_name, search_function in [
    ("Keyword Search", keyword_wrapper),
    ("Universal Keyword Search", universal_keyword_wrapper),
    ("Semantic Search", semantic_wrapper),
    ("Metadata + Keyword Search", metadata_keyword_wrapper),
    ("Metadata + Semantic Search", metadata_semantic_wrapper),
    ("Hybrid Search (60/40)", hybrid_wrapper),
]:
    result = await evaluate_search_method(
        method_name,
        search_function,
        eval_dataset.queries,
        eval_dataset.ground_truth,
    )
    evaluation_results.append(result)

print("\n✅ All evaluations complete!")


Evaluating: Keyword Search
  ✅ Completed: Precision=0.000, Recall=0.000

Evaluating: Universal Keyword Search
  ✅ Completed: Precision=0.000, Recall=0.000

Evaluating: Semantic Search


  ✅ Completed: Precision=0.010, Recall=0.024

Evaluating: Metadata + Keyword Search
  ✅ Completed: Precision=0.000, Recall=0.000

Evaluating: Metadata + Semantic Search


  ✅ Completed: Precision=0.019, Recall=0.040

Evaluating: Hybrid Search (60/40)


  ✅ Completed: Precision=0.010, Recall=0.024

✅ All evaluations complete!


## 5. Compare Results

Let's visualize and compare all search methods.

In [7]:
# Create comparison DataFrame
comparison_data = []
for eval_result in evaluation_results:
    method = eval_result["method"]
    results = eval_result["results"]

    comparison_data.append(
        {
            "Method": method,
            "Precision": results.precision,
            "Recall": results.recall,
            "F1 Score": results.f1_score,
            "MAP": results.map_score,
            "MRR": results.mrr_score,
            "NDCG": results.ndcg_score,
            "Hit Rate": results.hit_rate,
        }
    )

df = pd.DataFrame(comparison_data)
print("\n" + "=" * 100)
print("COMPARISON OF ALL SEARCH METHODS")
print("=" * 100)
print(df.to_string(index=False))
print("=" * 100)


COMPARISON OF ALL SEARCH METHODS
                    Method  Precision   Recall  F1 Score      MAP      MRR     NDCG  Hit Rate
            Keyword Search   0.000000 0.000000  0.000000 0.000000 0.000000 0.000000  0.000000
  Universal Keyword Search   0.000000 0.000000  0.000000 0.000000 0.000000 0.000000  0.000000
           Semantic Search   0.009524 0.023810  0.013605 0.011905 0.023810 0.018422  0.047619
 Metadata + Keyword Search   0.000000 0.000000  0.000000 0.000000 0.000000 0.000000  0.000000
Metadata + Semantic Search   0.019048 0.039683  0.025740 0.015079 0.033333 0.027066  0.095238
     Hybrid Search (60/40)   0.009524 0.023810  0.013605 0.011905 0.023810 0.018422  0.047619


In [8]:
# Visualize results when optional plotting packages are installed
if plt is None:
    print("\nInfo: Install matplotlib and seaborn to render evaluation charts.")
else:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    # 1. Precision, Recall, F1
    metrics_df = df[["Method", "Precision", "Recall", "F1 Score"]].set_index("Method")
    metrics_df.plot(kind="bar", ax=axes[0, 0], rot=45)
    axes[0, 0].set_title("Precision, Recall, and F1 Score", fontsize=14, fontweight="bold")
    axes[0, 0].set_ylabel("Score")
    axes[0, 0].legend(loc="lower right")
    axes[0, 0].set_ylim([0, 1])
    axes[0, 0].grid(axis="y", alpha=0.3)

    # 2. Rank-Aware Metrics
    rank_df = df[["Method", "MAP", "MRR", "NDCG"]].set_index("Method")
    rank_df.plot(kind="bar", ax=axes[0, 1], rot=45, color=["#2ecc71", "#3498db", "#e74c3c"])
    axes[0, 1].set_title("Rank-Aware Metrics (MAP, MRR, NDCG)", fontsize=14, fontweight="bold")
    axes[0, 1].set_ylabel("Score")
    axes[0, 1].legend(loc="lower right")
    axes[0, 1].set_ylim([0, 1])
    axes[0, 1].grid(axis="y", alpha=0.3)

    # 3. Hit Rate
    df.plot(
        x="Method",
        y="Hit Rate",
        kind="bar",
        ax=axes[1, 0],
        color="#9b59b6",
        rot=45,
        legend=False,
    )
    axes[1, 0].set_title(
        "Hit Rate (% Queries with ≥1 Relevant Result)", fontsize=14, fontweight="bold"
    )
    axes[1, 0].set_ylabel("Hit Rate")
    axes[1, 0].set_ylim([0, 1])
    axes[1, 0].grid(axis="y", alpha=0.3)

    # 4. Overall Comparison
    overall_scores = df[
        ["Precision", "Recall", "F1 Score", "MAP", "MRR", "NDCG", "Hit Rate"]
    ].mean()
    overall_scores.plot(kind="barh", ax=axes[1, 1], color="#34495e")
    axes[1, 1].set_title("Average Metric Scores Across All Methods", fontsize=14, fontweight="bold")
    axes[1, 1].set_xlabel("Average Score")
    axes[1, 1].set_xlim([0, 1])
    axes[1, 1].grid(axis="x", alpha=0.3)

    plt.tight_layout()
    plt.savefig("evaluation_results.png", dpi=300, bbox_inches="tight")
    print("\n✅ Visualization saved to 'evaluation_results.png'")
    plt.show()


ℹ️ Install matplotlib and seaborn to render evaluation charts.


## 6. Best Method Analysis

In [9]:
# Find best method for each metric
print("\n" + "=" * 80)
print("BEST METHOD FOR EACH METRIC")
print("=" * 80)

metrics = ["Precision", "Recall", "F1 Score", "MAP", "MRR", "NDCG", "Hit Rate"]
for metric in metrics:
    best_idx = df[metric].idxmax()
    best_method = df.loc[best_idx, "Method"]
    best_score = df.loc[best_idx, metric]
    print(f"  {metric:20s}: {best_method:30s} ({best_score:.4f})")

# Overall best method (highest average)
df["Average"] = df[metrics].mean(axis=1)
best_overall_idx = df["Average"].idxmax()
best_overall = df.loc[best_overall_idx, "Method"]
best_overall_score = df.loc[best_overall_idx, "Average"]

print("\n" + "=" * 80)
print(f"🏆 OVERALL BEST METHOD: {best_overall} (Avg: {best_overall_score:.4f})")
print("=" * 80)


BEST METHOD FOR EACH METRIC
  Precision           : Metadata + Semantic Search     (0.0190)
  Recall              : Metadata + Semantic Search     (0.0397)
  F1 Score            : Metadata + Semantic Search     (0.0257)
  MAP                 : Metadata + Semantic Search     (0.0151)
  MRR                 : Metadata + Semantic Search     (0.0333)
  NDCG                : Metadata + Semantic Search     (0.0271)
  Hit Rate            : Metadata + Semantic Search     (0.0952)

🏆 OVERALL BEST METHOD: Metadata + Semantic Search (Avg: 0.0365)


## 7. Interpretation Guide

In [10]:
print("\n" + "=" * 80)
print("METRIC INTERPRETATION GUIDE")
print("=" * 80)

interpretations = {
    "Precision": {
        "definition": "Ratio of relevant docs retrieved to total retrieved",
        "use_case": "Critical for LLMs with limited context length",
        "tip": "High precision reduces hallucinations",
        "threshold": 0.7,
    },
    "Recall": {
        "definition": "Ratio of relevant docs retrieved to total relevant docs",
        "use_case": "Generally useful across scenarios",
        "tip": "Recall of 0.5 means missing 50% of relevant documents",
        "threshold": 0.6,
    },
    "F1 Score": {
        "definition": "Harmonic mean of precision and recall",
        "use_case": "When you need balance between precision and recall",
        "tip": "Avoid if optimizing for precision OR recall independently",
        "threshold": 0.6,
    },
    "MAP": {
        "definition": "Mean average precision across queries (rank-aware)",
        "use_case": "When order of retrieval matters",
        "tip": "mAP of 0.2 means correct answers in top 5 on average",
        "threshold": 0.5,
    },
    "MRR": {
        "definition": "Average reciprocal rank of first relevant document",
        "use_case": "Multiple passages with same meaning",
        "tip": "Focuses on position of first relevant result",
        "threshold": 0.6,
    },
    "NDCG": {
        "definition": "Normalized discounted cumulative gain (position discount)",
        "use_case": "Understanding ranking quality",
        "tip": "Higher scores for relevant docs appearing earlier",
        "threshold": 0.7,
    },
    "Hit Rate": {
        "definition": "Percentage of queries with at least one relevant result",
        "use_case": "Ensuring basic retrieval coverage",
        "tip": "Low hit rate means expanding corpus or improving queries",
        "threshold": 0.8,
    },
}

for metric in metrics:
    info = interpretations[metric]
    avg_score = df[metric].mean()
    status = "✅ GOOD" if avg_score >= info["threshold"] else "⚠️  NEEDS IMPROVEMENT"

    print(f"\n{metric} ({status})")
    print(f"  Average Score: {avg_score:.4f} (threshold: {info['threshold']})")
    print(f"  Definition: {info['definition']}")
    print(f"  Use Case: {info['use_case']}")
    print(f"  💡 Tip: {info['tip']}")


METRIC INTERPRETATION GUIDE

Precision (⚠️  NEEDS IMPROVEMENT)
  Average Score: 0.0063 (threshold: 0.7)
  Definition: Ratio of relevant docs retrieved to total retrieved
  Use Case: Critical for LLMs with limited context length
  💡 Tip: High precision reduces hallucinations

Recall (⚠️  NEEDS IMPROVEMENT)
  Average Score: 0.0146 (threshold: 0.6)
  Definition: Ratio of relevant docs retrieved to total relevant docs
  Use Case: Generally useful across scenarios
  💡 Tip: Recall of 0.5 means missing 50% of relevant documents

F1 Score (⚠️  NEEDS IMPROVEMENT)
  Average Score: 0.0088 (threshold: 0.6)
  Definition: Harmonic mean of precision and recall
  Use Case: When you need balance between precision and recall
  💡 Tip: Avoid if optimizing for precision OR recall independently

MAP (⚠️  NEEDS IMPROVEMENT)
  Average Score: 0.0065 (threshold: 0.5)
  Definition: Mean average precision across queries (rank-aware)
  Use Case: When order of retrieval matters
  💡 Tip: mAP of 0.2 means correct an

## 8. Cleanup

In [11]:
# Close RAG system
await db.close()
print("\n✅ RAG system closed")
print("\n" + "=" * 80)
print("EVALUATION COMPLETE!")
print("=" * 80)
print("\nKey Takeaways:")
print("  1. All 6 search methods tested with comprehensive metrics")
print("  2. Evaluation dataset saved for reproducibility")
print("  3. Visualizations show method-by-method comparison")
print("  4. Use these metrics to optimize your RAG system")
print("\nNext Steps:")
print("  • Tune parameters for best-performing method")
print("  • Expand evaluation dataset with domain-specific queries")
print("  • Try ensemble methods combining multiple approaches")
print("  • Monitor metrics in production for continuous improvement")


✅ RAG system closed

EVALUATION COMPLETE!

Key Takeaways:
  1. All 6 search methods tested with comprehensive metrics
  2. Evaluation dataset saved for reproducibility
  3. Visualizations show method-by-method comparison
  4. Use these metrics to optimize your RAG system

Next Steps:
  • Tune parameters for best-performing method
  • Expand evaluation dataset with domain-specific queries
  • Try ensemble methods combining multiple approaches
  • Monitor metrics in production for continuous improvement
